## Graph traversal

Finding a path from the eyes (`ol_sensory`) to the motor neurons (`vnc_motor`)

The idea: the toy LIF in `01_lif.ipynb` only went one hop deep, so the charge had nowhere further to go
- instead of hand querying neighbours of neighbours, let a graph do the walking
- the graph structurally shows how neurons are connected, it does not simulate any charge moving
- the shortest path is a question of structure (is there a route?), LIF is a question of dynamics (does charge survive it?)

the goal is to find a real `ol_sensory` -> `vnc_motor` path
- then that path (and its neurons/edges) can be handed back to the LIF to see if the charge dies before reaching `vnc_motor`

### Init

In [1]:
import pandas as pd
from pathlib import Path

root = Path.cwd().parent
# root = Path(__file__).resolve().parent.parent

neurons_file = root / 'data' / 'body-annotations-male-cns-v1.0-minconf-0.5.feather'
connections_file = root / 'data' / 'connectome-weights-male-cns-v1.0-minconf-0.5.feather'
filtered_path = root / 'data' / 'pipeline_edges_filtered.feather'

df = pd.read_feather(neurons_file)
traced = df[df['status'] == 'Traced']

id_to_superclass = traced.set_index('bodyId')['superclass'].to_dict()

#### Loading the filtered pipeline edges

if this has been built before, load the cached feather file instead of rescanning the full 151M-row connections file

In [ ]:
if filtered_path.exists():
    strong_only = pd.read_feather(filtered_path)
else:
    # code from the exploration
    import pyarrow.feather as feather
    import duckdb

    # the whole table is needed, so read_table it is
    meta = feather.read_table(connections_file, memory_map=True)

    lookup = traced[['bodyId', 'superclass']] # data frame of 2 fields

    # the hypothesized pathway from eyes -> legs
    target_superclasses = [
        'ol_sensory', 'ol_intrinsic', 'visual_projection',
        'cb_intrinsic', 'descending_neuron',
        'vnc_intrinsic', 'vnc_motor'
    ]

    # again let duckdb do the heavy lifting
    # because it never materializes into 151M rows
    con = duckdb.connect()
    con.execute("PRAGMA memory_limit='6GB'")
    con.execute("PRAGMA threads=2")

    con.register('lookup', lookup)
    con.register('connections', meta)

    # dynamically build list of placeholders
    placeholders = ",".join(["?"] * len(target_superclasses))

    pipeline_edges = con.execute(f"""
        SELECT c.body_pre, c.body_post, c.weight,
            pre.superclass AS pre_sc, post.superclass AS post_sc
        FROM connections AS c
        JOIN lookup AS pre ON c.body_pre = pre.bodyId
        JOIN lookup AS post ON c.body_post = post.bodyId
        WHERE pre.superclass IN ({placeholders})
        AND post.superclass IN ({placeholders})
    """, target_superclasses + target_superclasses).df()

    print(pipeline_edges.shape)
    print(pipeline_edges.head())

    # same-superclass edges are local self-talk, not pipeline progress
    cross_region = pipeline_edges[pipeline_edges['pre_sc'] != pipeline_edges['post_sc']]

    # weight=1 correlated with unreliable/fragment-adjacent connections earlier
    strong_only = cross_region[cross_region['weight'] >= 3]

    print(strong_only.shape)

    strong_only.to_feather(root / 'data' / 'pipeline_edges_filtered.feather')

> `pipeline_edges` comes back very large (22237488 rows in the original run)
>
> same-superclass edges are dropped first, local self-talk isn't pipeline progress
>
> weight=1 edges are dropped too, correlated with unreliable/fragment-adjacent connections earlier
>
> the result is cached to `pipeline_edges_filtered.feather` so this expensive scan doesn't need to run again

In [3]:
strong_only_sorted = strong_only.sort_values('weight', ascending=False)
print(strong_only_sorted.head(20))

          body_pre  body_post  weight             pre_sc            post_sc
21628393     10066      10023    1554       ol_intrinsic  visual_projection
17542801     10309      10641    1439  visual_projection       cb_intrinsic
21628392     10066      10016    1386       ol_intrinsic  visual_projection
17570255     11041      10419    1373       ol_intrinsic  visual_projection
13410158     10231     522405    1367  visual_projection       cb_intrinsic
907429       10308      10025    1260       ol_intrinsic  visual_projection
17570254     11041      10034    1249       ol_intrinsic  visual_projection
1541452      13310      10019    1096       ol_intrinsic  visual_projection
21628391     10066      10015    1061       ol_intrinsic  visual_projection
779265       10248      11549    1056  visual_projection       cb_intrinsic
17570253     11041      10181    1048       ol_intrinsic  visual_projection
6343742      10436      10025    1026       ol_intrinsic  visual_projection
17523537    

#### Building the graph

why `networkx` specifically:

it's the simplest, best-documented option, but it's pure Python under the hood, loading the *full* 151M-edge connectome into it would likely crash again, the way the plain Python loop and the first DuckDB attempt did before.

`strong_only` is already filtered down from 151M to a much smaller edge count by this point though, small enough that `networkx`'s simplicity wins over faster but harder-to-use options like `igraph` or `networkit`, which would only be worth the extra learning curve if this filtered set still turned out too big to handle

In [4]:
import networkx as nx

print(strong_only.head())

    body_pre  body_post  weight        pre_sc            post_sc
5     141666      54596       4  ol_intrinsic  visual_projection
20    141770      41352      10    ol_sensory       ol_intrinsic
24    141787      19588       5  ol_intrinsic  visual_projection
26    141799      14446       5  cb_intrinsic  descending_neuron
30    141856      17405       5  ol_intrinsic  visual_projection


In [ ]:
# takes a pandas DataFrame where each row represents one edge, and builds an actual networkx graph
G = nx.from_pandas_edgelist(
    strong_only,            # df
    source='body_pre',      # which column holds the starting node of each edge
    target='body_post',     # which column holds the ending node of each edge
    edge_attr='weight',     # carry the weight column onto each edge as an attribute
    create_using=nx.DiGraph # directed graph because energy flows one way
)

print(G.number_of_nodes(), G.number_of_edges())

122393 1852081


> good news, the program doesn't crash building this graph

so the traversal can occur, but there's one problem: `nx.shortest_path(G, source, target, weight='weight')` doesn't know `weight` means "connection strength". graph problems are meant to minimize a weight, not maximize it, and what's actually wanted is the path through the biggest synapse counts

#### Weight to cost

transform each edge's weight into a cost where strong = cheap, weak = expensive
- a common, principled way to do this: cost = 1 / weight
- a weight-1500 edge becomes cost = 0.00067, a weight-3 edge becomes cost = 0.33

In [6]:
for u, v, data in G.edges(data=True):
    data['cost'] = 1 / data['weight']

#### First attempt

feed a real target bodyId whose superclass is `vnc_motor`, picked as the very first matching row

In [7]:
vnc_motor_neuron = traced.query('superclass == "vnc_motor"').iloc[0]
print(vnc_motor_neuron)

target_id = vnc_motor_neuron['bodyId']
print(target_id in G.nodes)

assignedOlHex1                       NaN
assignedOlHex2                       NaN
bodyId                            164190
flywireType                          NaN
group                           164190.0
instance                        MNad21_L
somaSide                               L
statusLabel        Prelim Roughly traced
superclass                     vnc_motor
type                              MNad21
vfbId                       VFB_jrmc20hz
hemibrainType                        NaN
itoleeHl                             NaN
supertype                            NaN
birthtime                          early
mancBodyid                       10482.0
mancGroup                        10473.0
mancType                          MNad21
subclass                              ad
synonyms                             NaN
class                                NaN
rootSide                             NaN
somaNeuromere                         A1
trumanHl                             TBD
dimorphism      

> Note:
>
> picked `vnc_motor_neuron = traced.query('superclass == "vnc_motor"').iloc[0]`, the first matching row, without checking what it actually was
>
> `bodyId` 164190, `type` `MNad21`, `somaNeuromere` A1, that's the first abdominal segment, not a leg
>
> this wasn't caught until much later, the whole path traced from this neuron turned out to end at an abdominal motor neuron, not a leg one, worth checking `somaNeuromere`/`exitNerve` before trusting a `.iloc[0]` pick like this again

assuming the start and end are both in the graph, this should create a path, note: this cell is expected to raise an error

In [8]:
try:
    path = nx.shortest_path(G, source=11139, target=target_id, weight='cost')
    print(path)
    print(len(path))
except nx.NetworkXNoPath as e:
    print("No path found:", e)

No path found: No path between 11139 and 164190.


> raises `networkx.exception.NetworkXNoPath: No path between 11139 and 164190`

check how much `11139` can even reach at all:

In [ ]:
# nx.descendants()
# Think of it as "everything downstream."
# It doesn't just give the nodes one edge away, it keeps following the graph outward
# and returns the complete set of everything it can eventually reach.
reachable = nx.descendants(G, 11139)
print(len(reachable))

5


> only reaches 5 other nodes total
>
> this matches the original `real_neighbours` cluster from `01_lif.ipynb`, which was also connected to only 5 nodes
>
> none of those 5 `ol_intrinsic` targets happen to have their own strong, cross-region outgoing edges that survived the filter, so the graph stops one hop out, same as the toy LIF simulation found by hand

try a different `ol_sensory` starting neuron instead:

In [10]:
ol_sensory_ids = traced.query('superclass == "ol_sensory"')['bodyId']
for nid in ol_sensory_ids[:20]:
    if nid in G.nodes:
        r = len(nx.descendants(G, nid))
        print(nid, r)

11139 5
14428 80350
15479 80350
15625 80350
16394 6
16398 80353
16681 80350
16924 5
17721 80350
18040 80352
18975 10
18979 80350
19418 5
20107 80350
20428 80350
21134 6
22842 80350
24385 6
25245 5
25360 80350


> `11139` can reach 5 nodes while most of the rest can reach 80,350
>
> confirms `11139` was just a poor, low-connectivity starting point, not a flaw in the graph, so the starting neuron will be changed

#### Trying again with a better-connected source

In [11]:
source_id = 14428 # this id reaches 80k nodes

> Note
>
> picked `source_id = 14428` for its connectivity (reaches 80k nodes), same as `11139` before it, without checking what kind of `ol_sensory` neuron it actually is
>
> given the `R1-R6`/`R7`/`R8`/`HBeyelet` split found earlier, this should have been checked with something like `traced.query('bodyId == 14428')[['type', 'class']]` before trusting it as a stand-in for "the eye", worth doing before relying on any path built from it

In [12]:
vnc_motor_ids = traced.query('superclass == "vnc_motor"')['bodyId']
valid_targets = [nid for nid in vnc_motor_ids if nid in G.nodes] # get vnc_motors present in the graph
print(len(valid_targets))

705


In [13]:
# starting from source_id, and following directed edges forward only,
# what's the complete set of every node it could possibly reach, through any number of hops?
reachable = nx.descendants(G, source_id)

In [14]:
# intersection of:
# "real motor neurons present in the graph" (valid_targets)
# "everything actually reachable from the specific starting neuron" (reachable)
reachable_motor = [nid for nid in valid_targets if nid in reachable]
print(len(reachable_motor))

702


In [15]:
path = nx.shortest_path(G, source=source_id, target=reachable_motor[0], weight='cost')
for nid in path:
    print(nid, id_to_superclass.get(nid))

14428 ol_sensory
13984 visual_projection
10286 descending_neuron
11974 cb_intrinsic
10680 descending_neuron
164190 vnc_motor


#### A non-linear surprise

the path comes back as `ol_sensory -> visual_projection -> descending_neuron -> cb_intrinsic -> descending_neuron -> vnc_motor`

`descending_neuron` to `cb_intrinsic` and then back to `descending_neuron`?
- brain tells body what to do, brain thinks? then to a different neuron, brain tells body what to do again
- it could be two different descending neurons, remember type vs. supertype
- it could be a real loop in the fly's brain biologically

new question: is the fly wiring not linear compared to expectation, or is there a flaw in the graph?

In [16]:
# check their types
weird = traced[['bodyId', 'superclass', 'subclass', 'class', 'type']].query('bodyId in [10286, 10680]')
print(weird)

     bodyId         superclass subclass class     type
267   10286  descending_neuron       xn   NaN  DNpe053
639   10680  descending_neuron       xn   NaN    DNp48


2 options to narrow it down:
- look up the neurons and see if the non-linear pathing makes biological sense
- check if this happens in any other path, to isolate whether it's a problem with the graph

loop through the first 5 reachable `vnc_motor` targets instead of just `reachable_motor[0]`, run the same shortest-path search against each one from the same `source_id`, and print each full path translated into superclasses, want to see if `descending_neuron` shows up twice with `cb_intrinsic` in between in most or all of these paths

In [17]:
for i in range(min(5, len(reachable_motor))):
    target = reachable_motor[i]
    path = nx.shortest_path(G, source=source_id, target=target, weight='cost')
    readable = [(nid, id_to_superclass.get(nid)) for nid in path]
    print(f"--- path to {target} ---")
    for nid, sc in readable:
        print(nid, sc)
    print()

--- path to 164190 ---
14428 ol_sensory
13984 visual_projection
10286 descending_neuron
11974 cb_intrinsic
10680 descending_neuron
164190 vnc_motor

--- path to 800056 ---
14428 ol_sensory
13984 visual_projection
10286 descending_neuron
15850 cb_intrinsic
11475 descending_neuron
803023 vnc_intrinsic
800056 vnc_motor

--- path to 800061 ---
14428 ol_sensory
13984 visual_projection
10648 cb_intrinsic
12885 descending_neuron
12612 cb_intrinsic
10056 descending_neuron
800286 vnc_intrinsic
800061 vnc_motor

--- path to 800146 ---
14428 ol_sensory
900035 visual_projection
11573 descending_neuron
11832 cb_intrinsic
10001 descending_neuron
800146 vnc_motor

--- path to 800158 ---
14428 ol_sensory
13984 visual_projection
10648 cb_intrinsic
12885 descending_neuron
12612 cb_intrinsic
10444 descending_neuron
800158 vnc_motor



> output is quite large, so to summarize: the fly brain is not as linear as expected, the double-descending-neuron pattern repeats

### Conclusion

what was the point of all this?
- the graph structurally shows how neurons are connected
- scaling the toy LIF would show the distribution of energy, answering: would the charge die before reaching `vnc_motor`?

so next would be to scale the LIF:
- tune `decay_factor`/`threshold`/`external_input` just enough to see whether the real path can physically carry a signal end to end